In [1]:
import os
import sys
from pathlib import Path

In [2]:
import sys
import subprocess

# Install required packages from the notebook using the current Python executable
required = [
    "langchain",
    "langchain-core",
    "langchain-text-splitters",
    "langchain-ollama",
    "langchain-community",
    "faiss-cpu",
    "rank-bm25",
    "pypdf",
]

def install(packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '--upgrade'] + packages
    print('Running:', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('ERROR:', result.stderr)
    return result.returncode

# Attempt install
rc = install(required)
if rc == 0:
    print('Install completed successfully.')
else:
    print('One or more packages failed to install. Check errors above.')

Running: c:\Users\arpit\anaconda3\python.exe -m pip install --upgrade langchain langchain-core langchain-text-splitters langchain-ollama langchain-community faiss-cpu rank-bm25 pypdf

Install completed successfully.

Install completed successfully.


In [3]:
import sys, traceback

print("Executing with Python Version:", sys.version)
print("Verifying library imports...")

checks = [
    ("langchain_community.document_loaders.PyPDFLoader", "from langchain_community.document_loaders import PyPDFLoader"),
    ("langchain_text_splitters.RecursiveCharacterTextSplitter", "from langchain_text_splitters import RecursiveCharacterTextSplitter"),
    ("langchain_community.vectorstores.FAISS", "from langchain_community.vectorstores import FAISS"),
    ("langchain_ollama.OllamaEmbeddings, ChatOllama", "from langchain_ollama import OllamaEmbeddings, ChatOllama"),
    ("langchain_community.retrievers.BM25Retriever", "from langchain_community.retrievers import BM25Retriever"),
    ("langchain.retrievers.EnsembleRetriever", "from langchain.retrievers import EnsembleRetriever"),
    ("langchain_core.prompts.ChatPromptTemplate", "from langchain_core.prompts import ChatPromptTemplate"),
    ("langchain_core.runnables.RunnablePassthrough", "from langchain_core.runnables import RunnablePassthrough"),
    ("langchain_core.output_parsers.StrOutputParser", "from langchain_core.output_parsers import StrOutputParser"),
]

missing = []
for name, stmt in checks:
    try:
        exec(stmt, globals())
        print(f"OK: {name}")
    except Exception as e:
        tb = traceback.format_exc()
        missing.append((name, str(e), tb))
        print(f"MISSING: {name} -> {e}")

if missing:
    print('\nSummary of missing imports:')
    for name, err, tb in missing:
        print(f'- {name}: {err}')
    print('\nNotes:')
    print('- `FAISS` (faiss-cpu) is not available via pip on some Windows/python combos. Use conda: `conda install -c conda-forge faiss-cpu` or run on Linux.')
    print('- If `langchain.retrievers` or other langchain submodules are missing, try restarting the kernel and ensuring the notebook is using the same Python interpreter used for pip installs (sys.executable).')
    print('- To force-reinstall langchain:')
    print(f'    {sys.executable} -m pip install --upgrade langchain langchain-core langchain-community')
else:
    print('\nAll imports available.')


Executing with Python Version: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
Verifying library imports...


<string>:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.


OK: langchain_community.document_loaders.PyPDFLoader
OK: langchain_text_splitters.RecursiveCharacterTextSplitter
OK: langchain_community.vectorstores.FAISS
OK: langchain_ollama.OllamaEmbeddings, ChatOllama
OK: langchain_community.retrievers.BM25Retriever
MISSING: langchain.retrievers.EnsembleRetriever -> No module named 'langchain.retrievers'
OK: langchain_core.prompts.ChatPromptTemplate
OK: langchain_core.runnables.RunnablePassthrough
OK: langchain_core.output_parsers.StrOutputParser

Summary of missing imports:
- langchain.retrievers.EnsembleRetriever: No module named 'langchain.retrievers'

Notes:
- `FAISS` (faiss-cpu) is not available via pip on some Windows/python combos. Use conda: `conda install -c conda-forge faiss-cpu` or run on Linux.
- If `langchain.retrievers` or other langchain submodules are missing, try restarting the kernel and ensuring the notebook is using the same Python interpreter used for pip installs (sys.executable).
- To force-reinstall langchain:
    c:\Users\

In [4]:
# Retriever helper: prefer EnsembleRetriever class if available, otherwise fall back to BM25Retriever class
try:
    from langchain.retrievers import EnsembleRetriever
    retriever_class = EnsembleRetriever
    print("EnsembleRetriever is available (class).")
except Exception:
    from langchain_community.retrievers import BM25Retriever
    retriever_class = BM25Retriever
    print("EnsembleRetriever not available — using BM25Retriever as fallback (class).")

# Expose a simple accessor: users should instantiate with the appropriate required args (docs, etc.)
def get_retriever_class():
    """Return the retriever class to instantiate later with proper arguments."""
    return retriever_class

# Example: show the class but do not instantiate (avoids validation errors)
print("Retriever class:", retriever_class)

EnsembleRetriever not available — using BM25Retriever as fallback (class).
Retriever class: <class 'langchain_community.retrievers.bm25.BM25Retriever'>


In [5]:

DATA_DIR = Path("books/pdf")
DATA_DIR.mkdir(exist_ok=True)

def ingest_ncert_pdfs(data_path: Path):
    """
    Scans the data directory, extracts text from PDF textbooks page-by-page,
    and returns a structured list of LangChain Document objects.
    """
    pdf_paths = list(data_path.glob("*.pdf"))
    if not pdf_paths:
        print(f"Warning: No PDF files found in {data_path.absolute()}. Please upload textbooks.")
        return
    
    parsed_pages = []
    for pdf_file in pdf_paths:
        print(f"Ingesting: {pdf_file.name}")
        # PyPDFLoader parses page-by-page to preserve structure and coordinate page-level citations
        loader = PyPDFLoader(str(pdf_file))
        try:
            pages = loader.load()
            parsed_pages.extend(pages)
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
        
    print(f"Successfully processed {len(parsed_pages)} total pages.")
    return parsed_pages

# Execute the ingestion pipeline
raw_pages = ingest_ncert_pdfs(DATA_DIR)

Ingesting: jeff101.pdf
Ingesting: jeff102.pdf
Ingesting: jeff102.pdf
Ingesting: jeff103.pdf
Ingesting: jeff104.pdf
Ingesting: jeff103.pdf
Ingesting: jeff104.pdf
Ingesting: jeff105.pdf
Ingesting: jeff106.pdf
Ingesting: jeff107.pdf
Ingesting: jeff105.pdf
Ingesting: jeff106.pdf
Ingesting: jeff107.pdf
Ingesting: jeff108.pdf
Ingesting: jeff109.pdf
Ingesting: jeff108.pdf
Ingesting: jeff109.pdf
Ingesting: jeff1ps.pdf
Ingesting: jeff1ps.pdf
Ingesting: jefp101.pdf
Ingesting: jefp101.pdf
Ingesting: jefp102.pdf
Ingesting: jefp102.pdf
Ingesting: jefp103.pdf
Ingesting: jefp103.pdf
Ingesting: jefp104.pdf
Ingesting: jefp104.pdf
Ingesting: jefp105.pdf
Ingesting: jefp105.pdf
Ingesting: jefp106.pdf
Ingesting: jefp106.pdf
Ingesting: jefp107.pdf
Ingesting: jefp107.pdf
Ingesting: jefp108.pdf
Ingesting: jefp108.pdf
Ingesting: jefp109.pdf
Ingesting: jefp109.pdf
Ingesting: jefp1ps.pdf
Ingesting: jefp1ps.pdf
Ingesting: jehp101.pdf
Ingesting: jehp101.pdf
Ingesting: jehp102.pdf
Ingesting: jehp102.pdf
Ingesting: 

In [6]:

def segment_textbook_pages(pages, chunk_size=512, chunk_overlap=75):
    """
    Splits long textbook pages into smaller segments using recursive splitting,
    preserving structural boundaries and context overlaps.
    """
    if not pages:
        print("Error: No documents found to split.")
        return
        
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        add_start_index=True
    )
    segmented_chunks = splitter.split_documents(pages)
    print(f"Successfully generated {len(segmented_chunks)} chunks.")
    return segmented_chunks

if raw_pages:
    processed_chunks = segment_textbook_pages(raw_pages)

Successfully generated 20315 chunks.


In [9]:
OLLAMA_EMBEDDING_MODEL = "nomic-embed-text"
INDEX_PERSIST_DIR = "./local_faiss_index"

def build_retrieval_network(chunks):
    """
    Creates parallel dense (FAISS) and sparse (BM25) search indexes,
    then combines them using LangChain's EnsembleRetriever and Reciprocal Rank Fusion.
    """
    # 1. Initialize local embeddings via Ollama
    print(f"Initializing local embeddings using model: '{OLLAMA_EMBEDDING_MODEL}'...")
    embeddings = OllamaEmbeddings(model=OLLAMA_EMBEDDING_MODEL)
    
    # 2. Build and save the dense FAISS vector index in BATCHES
    print(f"Building dense FAISS index from {len(chunks)} chunks in batches...")
    
    dense_db = None
    batch_size = 50  # Smaller batch size to prevent Ollama from crashing
    
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i : i + batch_size]
        print(f" -> Embedding batch {i} to {i + len(batch)}...")
        if dense_db is None:
            dense_db = FAISS.from_documents(batch, embeddings)
        else:
            dense_db.add_documents(batch)
            
    dense_db.save_local(INDEX_PERSIST_DIR)
    dense_retriever = dense_db.as_retriever(search_kwargs={"k": 3})
    
    # 3. Build the sparse BM25 index
    print("Building sparse BM25 index...")
    sparse_retriever = BM25Retriever.from_documents(chunks)
    sparse_retriever.k = 3
    
    # 4. Attempt to combine the retrievers using RRF
    try:
        from langchain.retrievers import EnsembleRetriever
        print("Configuring EnsembleRetriever...")
        final_retriever = EnsembleRetriever(
            retrievers=[dense_retriever, sparse_retriever],
            weights=[0.5, 0.5]
        )
        print("Hybrid retrieval network is active.")
    except ImportError:
        print("EnsembleRetriever not available. Falling back to dense FAISS retriever only.")
        final_retriever = dense_retriever
        
    return final_retriever

if raw_pages:
    hybrid_retriever = build_retrieval_network(processed_chunks)

Initializing local embeddings using model: 'nomic-embed-text'...
Building dense FAISS index from 20315 chunks in batches...
 -> Embedding batch 0 to 50...
 -> Embedding batch 50 to 100...
 -> Embedding batch 100 to 150...
 -> Embedding batch 150 to 200...
 -> Embedding batch 200 to 250...
 -> Embedding batch 250 to 300...
 -> Embedding batch 300 to 350...
 -> Embedding batch 350 to 400...
 -> Embedding batch 400 to 450...
 -> Embedding batch 450 to 500...
 -> Embedding batch 500 to 550...
 -> Embedding batch 550 to 600...
 -> Embedding batch 600 to 650...
 -> Embedding batch 650 to 700...
 -> Embedding batch 700 to 750...
 -> Embedding batch 750 to 800...
 -> Embedding batch 800 to 850...
 -> Embedding batch 850 to 900...
 -> Embedding batch 900 to 950...
 -> Embedding batch 950 to 1000...
 -> Embedding batch 1000 to 1050...
 -> Embedding batch 1050 to 1100...
 -> Embedding batch 1100 to 1150...
 -> Embedding batch 1150 to 1200...
 -> Embedding batch 1200 to 1250...
 -> Embedding batch

In [11]:
OLLAMA_GENERATIVE_MODEL = "llama3.2"

def construct_curriculum_chain(retriever):
    """
    Sets up the generation chain by combining the hybrid retriever,
    strict prompt templates, and the local Ollama language model.
    """
    # Initialize the local model
    local_llm = ChatOllama(model=OLLAMA_GENERATIVE_MODEL, temperature=0.1)
    
    # Configure a strict, grounding system prompt
    prompt_template = """
    You are an expert curriculum assistant for the Indian K-12 education system. 
    Your answers must be grounded strictly in the provided NCERT textbook context.
    If the context does not contain the answer, state that you do not have sufficient information.
    Do not use outside knowledge or introduce external syllabus topics.

    Retrieved Context:
    \"\"\"{context}\"\"\"

    Question: {question}

    Provide a clear, age-appropriate academic explanation with citations in step-by-step prose.
    """
    prompt = ChatPromptTemplate.from_template(prompt_template)
    
    # Format retrieved contexts into a single block with source tracking
    def format_docs(docs):
        formatted_blocks = []  # FIXED: Added the empty list here
        for i, doc in enumerate(docs):
            source_file = os.path.basename(doc.metadata.get("source", "Unknown PDF"))
            page_number = doc.metadata.get("page", "N/A")
            formatted_blocks.append(
                f"Source {i+1}: {source_file} (Page {page_number})\nContent: {doc.page_content}"
            )
        return "\n\n".join(formatted_blocks)
    
    # Define the execution chain
    execution_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt
        | local_llm
        | StrOutputParser()
    )
    return execution_chain

if raw_pages:
    curriculum_chain = construct_curriculum_chain(hybrid_retriever)

In [16]:

def evaluate_assistant_run(query_text, retriever, chain):
    """
    Submits a query to the retrieval-generation chain, prints the output,
    and lists the exact sources retrieved by the system.
    """
    # Retrieve relevant document segments
    retrieved_segments = retriever.invoke(query_text)
    
    # Generate the grounded answer
    response = chain.invoke(query_text)
    
    # Display the results
    print("=" * 80)
    print(f"STUDENT QUESTION: {query_text}")
    print("=" * 80)
    print(f"GENERATED TUTOR ANSWER:\n{response}")
    print("=" * 80)
    print("VERIFIED EXCERPT SOURCES:")
    for idx, doc in enumerate(retrieved_segments):
        file_name = os.path.basename(doc.metadata.get('source', 'Syllabus File'))
        page_id = doc.metadata.get('page', 'N/A')
        print(f"- Source [{idx+1}]: {file_name} | Page Reference: {page_id}")
    print("=" * 80)

if raw_pages:
    # Execute a sample test query
    sample_query = "what is quadratic equation? "
    evaluate_assistant_run(sample_query, hybrid_retriever, curriculum_chain)

STUDENT QUESTION: what is quadratic equation? 
GENERATED TUTOR ANSWER:
A quadratic equation is defined as an equation of the form ax2 + bx + c = 0, where a, b, and c are real numbers, and a ≠ 0.

According to Source 1 (Page 1), "A quadratic equation in the variable x is an equation of the form ax2 + bx + c = 0, where a, b, c are real numbers, a  0."

This definition implies that a quadratic equation has three main components: 

1. A squared term (ax2)
2. A linear term (bx)
3. A constant term (c)

These components are connected by an equals sign (=) and the entire expression is set equal to zero (0).

For example, consider the equation 2x2 + x - 300 = 0. This is a quadratic equation because it follows the standard form ax2 + bx + c = 0.

In Source 2 (Page 1), it's mentioned that "any equation of the form p(x) = 0, where p(x) is a polynomial of degree 2, is a quadratic equation." However, this definition is more general and doesn't provide specific details about the coefficients a, b, a